<a href="https://colab.research.google.com/github/freyasheth/smart-home-energy-prediction/blob/main/smart_home_energy_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow import keras

from sklearn.model_selection import train_test_split
from tqdm import tqdm
import os
import librosa
import librosa.display
import h5py
import warnings
warnings.filterwarnings('ignore')

from joblib import Parallel, delayed

In [ ]:
import kagglehub

path = kagglehub.dataset_download("mexwell/smart-home-energy-consumption")

print("Path to dataset files:", path)

100%|██████████| 1.25M/1.25M [00:00<00:00, 2.36MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mexwell/smart-home-energy-consumption/versions/1


In [ ]:
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("mexwell/smart-home-energy-consumption")
print(f"Active dataset path: {path}")

files = os.listdir(path)
print(f"Files found in directory: {files}")

csv_filename = None
for file in files:
    if file.endswith('.csv'):
        csv_filename = file
        break

if csv_filename:
    full_path = os.path.join(path, csv_filename)

    df = pd.read_csv(full_path)
    print(f"\n Successfully loaded: {csv_filename}")

    display(df.head())
else:
    print("\n No CSV file was found in that directory.")

Using Colab cache for faster access to the 'smart-home-energy-consumption' dataset.
Active dataset path: /kaggle/input/smart-home-energy-consumption
Files found in directory: ['smart_home_energy_consumption_large.csv', '.nfs000000006f3b4fa000000115']

✅ Successfully loaded: smart_home_energy_consumption_large.csv


,Home ID,Appliance Type,Energy Consumption (kWh),Time,Date,Outdoor Temperature (°C),Season,Household Size
0,94,Fridge,0.20,21:12,2023-12-02,-1.0,Fall,2
1,435,Oven,0.23,20:11,2023-08-06,31.1,Summer,5
2,466,Dishwasher,0.32,06:39,2023-11-21,21.3,Fall,3
3,496,Heater,3.92,21:56,2023-01-21,-4.2,Winter,1
4,137,Microwave,0.44,04:31,2023-08-26,34.5,Summer,5


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load and prepare the timeline
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df = df.sort_values('Datetime').reset_index(drop=True)

# For this example, let's focus on a single home to mimic the paper's single-house setup
# Alternatively, you could aggregate total consumption per hour/day across all homes.
home_df = df[df['Home ID'] == 94].copy()
home_df.set_index('Datetime', inplace=True)

# 2. Select features and target
# Target: Energy Consumption (kWh)
# Features: Outdoor Temperature, Household Size, Appliance Type, Season
features = ['Outdoor Temperature (°C)', 'Household Size', 'Appliance Type', 'Season']
target = ['Energy Consumption (kWh)']

# 3. Scale numerical data and One-Hot Encode categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), ['Outdoor Temperature (°C)', 'Household Size']),
        ('cat', OneHotEncoder(sparse_output=False), ['Appliance Type', 'Season'])
    ])

X_scaled = preprocessor.fit_transform(home_df[features])
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(home_df[target])

# Combine back into a single array for sequence generation
processed_data = np.hstack((X_scaled, y_scaled))

In [ ]:
def create_sequences(data, target_col_index, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback), :])
        y.append(data[i + lookback, target_col_index])
    return np.array(X), np.array(y)

# Define the lookback window (e.g., past 10 events/timesteps to predict the next)
LOOKBACK = 10
target_idx = processed_data.shape[1] - 1 # The last column is our target

X, y = create_sequences(processed_data, target_idx, LOOKBACK)

# Split into training and testing sets (80% train, 20% test)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout

# Number of features in our processed dataset
n_features = X_train.shape[2]

model = Sequential()

# CNN Layers (Feature Extraction)
# Filter num: 64, 128, 256, 128, 64. Kernel size: 3, Strides: 1, Activation: Relu
model.add(Conv1D(filters=64, kernel_size=3, strides=1, activation='relu', padding='same', input_shape=(LOOKBACK, n_features)))
model.add(Conv1D(filters=128, kernel_size=3, strides=1, activation='relu', padding='same'))
model.add(Conv1D(filters=256, kernel_size=3, strides=1, activation='relu', padding='same'))
model.add(Conv1D(filters=128, kernel_size=3, strides=1, activation='relu', padding='same'))
model.add(Conv1D(filters=64, kernel_size=3, strides=1, activation='relu', padding='same'))

# LSTM Layers (Temporal Sequence Learning)
# Num: 64, 128, 256, 128, 64. Activation: Tanh
model.add(LSTM(64, activation='tanh', return_sequences=True))
model.add(LSTM(128, activation='tanh', return_sequences=True))
model.add(LSTM(256, activation='tanh', return_sequences=True))
model.add(LSTM(128, activation='tanh', return_sequences=True))
model.add(LSTM(64, activation='tanh', return_sequences=True))
# The paper mentions a 6th LSTM layer with "n_feats" units
model.add(LSTM(n_features, activation='tanh', return_sequences=False))

# Dropout & Output
model.add(Dropout(0.1))
# Output layer to predict the single continuous energy consumption value
model.add(Dense(1, activation='linear'))

# The paper specifies the Adam optimizer and Mean Squared Error (MSE) loss
model.compile(optimizer='adam', loss='mse')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 10, 64)         │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 10, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 10, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 10, 128)        │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 10, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 10, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 10, 256)        │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 10, 128)        │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 10, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 17)             │         5,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 17)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027,866 (3.92 MB)

 Trainable params: 1,027,866 (3.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model (The paper uses 600 epochs and batch size of 16)
history = model.fit(
    X_train, y_train,
    epochs=600,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

# Generate Predictions
predictions_scaled = model.predict(X_test)

# Inverse transform to get actual kWh values back
predictions = y_scaler.inverse_transform(predictions_scaled)
y_test_actual = y_scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate error metrics as done in the paper
mse = mean_squared_error(y_test_actual, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_actual, predictions)

Epoch 1/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 187ms/step - loss: 0.1070 - val_loss: 0.0736
Epoch 2/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0780 - val_loss: 0.0600
Epoch 3/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0651 - val_loss: 0.0554
Epoch 4/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0688 - val_loss: 0.0555
Epoch 5/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0681 - val_loss: 0.0555
Epoch 6/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0686 - val_loss: 0.0557
Epoch 7/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0653 - val_loss: 0.0565
Epoch 8/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0632 - val_loss: 0.0562
Epoch 9/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0639 - val_loss: 0.0581
Epoch 10/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0635 - val_loss: 0.0558
Epoch 11/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0625 - val_loss: 0.0630
Epoch 12/600
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/st

In [ ]:
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Square Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")

Mean Squared Error (MSE): 2.7551
Root Mean Square Error (RMSE): 1.6598
Mean Absolute Error (MAE): 1.2967


In [ ]:
len(df["Home ID"].unique())

500

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "/kaggle/working/best_model.keras",
    save_best_only=True,
    monitor="val_loss"
)

In [ ]:
import os
import pandas as pd
import numpy as np
import kagglehub
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout

# Let's start by merging the Date and Time columns into a proper Datetime format.
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])

# Sort everything first by the specific home, and then by time.
# This ensures our time-series sequences flow naturally.
df = df.sort_values(['Home ID', 'Datetime']).reset_index(drop=True)

# Setting up what we want to use to predict (features) and what we're predicting (target)
features = ['Outdoor Temperature (°C)', 'Household Size', 'Appliance Type', 'Season']
target = ['Energy Consumption (kWh)']

# We need to scale numbers so they don't overwhelm the neural net, and one-hot encode categories.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), ['Outdoor Temperature (°C)', 'Household Size']),
        ('cat', OneHotEncoder(sparse_output=False), ['Appliance Type', 'Season'])
    ])
preprocessor.fit(df[features])

# We also scale the target variable separately so we can easily un-scale our predictions later
y_scaler = MinMaxScaler()
y_scaler.fit(df[target])

# A helper function to create sliding windows of data for our time-series model
def create_sequences(data, target_col_index, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback), :])
        y.append(data[i + lookback, target_col_index])
    return np.array(X), np.array(y)

a = 10 # This is our lookback window (how many previous steps we look at)
X_all, y_all = [], []

# Process each house individually. If we didn't do this, a sequence might accidentally grab the end of Home A's data to predict the start of Home B's data!
for home_id in df['Home ID'].unique():
    home_df = df[df['Home ID'] == home_id].copy()
    home_df.set_index('Datetime', inplace=True)
    home_df = home_df.sort_index()

    # Transform our features and target using the scalers we fit earlier
    X_transformed = preprocessor.transform(home_df[features])
    y_transformed = y_scaler.transform(home_df[target])

    # Stick them together into one big array so we can slice it into sequences
    processed_data = np.hstack((X_transformed, y_transformed))
    target_idx = processed_data.shape[1] - 1

    # Chop the data into sequences for this specific house
    X_home, y_home = create_sequences(processed_data, target_idx, a)

    # Only append if we actually generated sequences (in case a house had too little data)
    if len(X_home) > 0:
        X_all.append(X_home)
        y_all.append(y_home)

# Combine all the individual house sequences back into one massive dataset
X = np.vstack(X_all)
y = np.concatenate(y_all)

# Standard 80/20 split for training and testing.
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

n_features = X_train.shape[2]

# Time to build the model
model = Sequential()

b = 64   # Base filter count
c = 128  # Mid filter count
d = 256  # Max filter count
e = 3    # Kernel size
f = 1    # Strides / Dense output size

# CNN Block
# These layers act as feature extractors, looking for localized patterns or spikes in the data
model.add(Conv1D(filters=b, kernel_size=e, strides=f, activation='relu', padding='same', input_shape=(a, n_features)))
model.add(Conv1D(filters=c, kernel_size=e, strides=f, activation='relu', padding='same'))
model.add(Conv1D(filters=d, kernel_size=e, strides=f, activation='relu', padding='same'))
model.add(Conv1D(filters=c, kernel_size=e, strides=f, activation='relu', padding='same'))
model.add(Conv1D(filters=b, kernel_size=e, strides=f, activation='relu', padding='same'))

#  LSTM Block
# LSTMs are great at remembering long-term temporal trends over our sequence window
model.add(LSTM(b, activation='tanh', return_sequences=True))
model.add(LSTM(c, activation='tanh', return_sequences=True))
model.add(LSTM(d, activation='tanh', return_sequences=True))
model.add(LSTM(c, activation='tanh', return_sequences=True))
model.add(LSTM(b, activation='tanh', return_sequences=True))
model.add(LSTM(n_features, activation='tanh', return_sequences=False))

# Drop 10% of connections to help prevent overfitting
model.add(Dropout(0.1))

# Linear output since we're predicting a continuous number (kWh)
model.add(Dense(f, activation='linear'))

# Compile with Adam optimizer and Mean Squared Error for regression
model.compile(optimizer='adam', loss='mse')

g = 50  # Epochs
h = 16  # Batch size

# Let's train it
history = model.fit(
    X_train, y_train,
    epochs=g,
    batch_size=h,
    validation_data=(X_test, y_test),
    verbose=1
)

# Grab our predictions for the test set
predictions_scaled = model.predict(X_test)

# The predictions are still scaled between 0 and 1, so we reverse the math to get real kWh values
predictions = y_scaler.inverse_transform(predictions_scaled)
y_test_actual = y_scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate standard error metrics to see how well we did
mse = mean_squared_error(y_test_actual, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_actual, predictions)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Epoch 1/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 84s 16ms/step - loss: 0.0585 - val_loss: 0.0582
Epoch 2/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 76s 16ms/step - loss: 0.0580 - val_loss: 0.0582
Epoch 3/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 80s 17ms/step - loss: 0.0579 - val_loss: 0.0581
Epoch 4/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 76s 16ms/step - loss: 0.0579 - val_loss: 0.0581
Epoch 5/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 77s 16ms/step - loss: 0.0579 - val_loss: 0.0581
Epoch 6/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 81s 16ms/step - loss: 0.0579 - val_loss: 0.0582
Epoch 7/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 77s 16ms/step - loss: 0.0579 - val_loss: 0.0581
Epoch 8/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 76s 16ms/step - loss: 0.0578 - val_loss: 0.0582
Epoch 9/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 76s 16ms/step - loss: 0.0578 - val_loss: 0.0582
Epoch 10/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 77s 16ms/step - loss: 0.0579 - val_loss: 0.0582
Epoch 11/50
4750/4750 ━━━━━━━━━━━━━━━━━━━━ 78s 16ms/step - loss: 0.0579 - val_loss: 0.0581
Epoch 12

MSE: 1.3960
RMSE: 1.1815
MAE: 0.8915

In [ ]:
import joblib

# let's save our fully trained neural network. The .keras format is the modern standard for saving TensorFlow/Keras models, packaging the architecture, weights, and training configuration all into one neat file.
model.save("cnn_lstm_energy_model.keras")

# we must save our data preprocessing tools.
# We use joblib here because it's highly efficient for saving scikit-learn objects (like our ColumnTransformer).
joblib.dump(preprocessor, "preprocessor.pkl")

# Similarly, we need to save the scaler we used for our target variable.
# We'll need this later to translate the model's scaled predictions (numbers between 0 and 1) back into actual, understandable energy values (kWh).
joblib.dump(y_scaler, "y_scaler.pkl")

print("Saved model, preprocessor, and target scaler successfully.")

Saved model, preprocessor, and target scaler successfully.


In [ ]:
import json

with open("history.json", "w") as f:
    json.dump(history.history, f)

In [ ]:
from google.colab import files

files.download("cnn_lstm_energy_model.keras")
files.download("preprocessor.pkl")
files.download("y_scaler.pkl")
files.download("history.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

> ### Above given downloaded files' Breakdown:
>
> Here is an overview of what each downloaded file does:
>
> * **`cnn_lstm_energy_model.keras` (The Brain)**
>     * This is the fully trained neural network. The `.keras` format contains the entire architecture (the 5 Conv1D layers and 6 LSTM layers), the compiled state (optimizer and loss function), and all the learned weights and biases. The function to load this `tensorflow.keras.models.load_model()`.
>
> * **`preprocessor.pkl` (The Translator)**
>     * This is a fitted `ColumnTransformer` (containing the `MinMaxScaler` and `OneHotEncoder`). A neural network expects new data to be scaled and encoded perfectly to match the training data. If we feed raw numbers (like an outdoor temp of 32°C) directly into the `.keras` model, it will break or give garbage predictions. We use joblib to load this in order to tranform any incoming API requests
>
> * **`y_scaler.pkl` (The Interpreter)**
>     * Because the model was trained on scaled target data (values between 0 and 1), its raw predictions will also be between 0 and 1. This scaler is required to apply `inverse_transform()` to the model's output, translating that arbitrary decimal back into a real-world, readable **kWh** value.
>
> * **`history.json` (The Logbook)**
>     * This contains the epoch-by-epoch training logs (loss, validation loss, MSE, etc.). While not needed for deployment, it is highly useful for generating Matplotlib or Seaborn charts for presentations or reports to prove that the model converged properly without overfitting.